In [1]:
!wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_45/gencode.v45.annotation.gtf.gz

--2024-11-14 08:02:02--  https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_45/gencode.v45.annotation.gtf.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49770653 (47M) [application/x-gzip]
Saving to: ‘gencode.v45.annotation.gtf.gz’

gencode.v45.annotat 100%[===================>]  47.46M  40.8MB/s    in 1.2s    

2024-11-14 08:02:04 (40.8 MB/s) - ‘gencode.v45.annotation.gtf.gz’ saved [49770653/49770653]



In [3]:
!gunzip gencode.v45.annotation.gtf.gz

In [1]:
import pandas as pd 

from dataclasses import dataclass, field

@dataclass
class GFFMeta:
    tags: set[str] = field(default_factory=set)
    attrs: dict[str, str] = field(default_factory=dict)

    @classmethod
    def from_str(cls, s: str):
        self = cls() 
        fields = s.split("; ")
        for field in fields:
            if field.startswith("tag "):
                self.tags.add(field[4:]) # remove 'tag '
            else:
                key, value = field.split(" ", maxsplit=1)
                self.attrs[key] = value
        return self
                

In [2]:
df = pd.read_table("gencode.v45.annotation.gtf", 
                   sep="\t", 
                   comment="#",
                   header=None,
                   names=["chrom",
                          "source",
                          "feature",
                          "start",
                          "end",
                          "score",
                          "strand",
                          "fname",
                          "attribute"])

In [3]:
df.head()

,chrom,source,feature,start,end,score,strand,fname,attribute
0,chr1,HAVANA,gene,11869,14409,.,+,.,"gene_id ""ENSG00000290825.1""; gene_type ""lncRNA..."
1,chr1,HAVANA,transcript,11869,14409,.,+,.,"gene_id ""ENSG00000290825.1""; transcript_id ""EN..."
2,chr1,HAVANA,exon,11869,12227,.,+,.,"gene_id ""ENSG00000290825.1""; transcript_id ""EN..."
3,chr1,HAVANA,exon,12613,12721,.,+,.,"gene_id ""ENSG00000290825.1""; transcript_id ""EN..."
4,chr1,HAVANA,exon,13221,14409,.,+,.,"gene_id ""ENSG00000290825.1""; transcript_id ""EN..."


In [4]:
genes = df[df['feature'] == 'gene'].copy()

In [5]:
genes['attribute'] = genes['attribute'].apply(GFFMeta.from_str)

In [6]:
genes['type'] = genes['attribute'].apply(lambda x: x.attrs['gene_type'])

In [7]:
genes['start'] = genes['start'] - 1

In [8]:
genes[['chrom', 'start', 'end', 'strand', 'type']].to_csv('genes.bed', sep="\t", index=False)

In [10]:
!head genes.bed

chrom	start	end	strand	type
chr1	11868	14409	+	"""lncRNA"""
chr1	12009	13670	+	"""transcribed_unprocessed_pseudogene"""
chr1	14695	24886	-	"""unprocessed_pseudogene"""
chr1	17368	17436	-	"""miRNA"""
chr1	29553	31109	+	"""lncRNA"""
chr1	30365	30503	+	"""miRNA"""
chr1	34553	36081	-	"""lncRNA"""
chr1	52472	53312	+	"""unprocessed_pseudogene"""
chr1	57597	64116	+	"""lncRNA"""


In [ ]:

formatted_blat